# EAP

### Load and Convert Fine-tuned Transformer Model to TransformerLens

In [6]:
from src import load_finetuned_model

base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")

model.eval()

Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


Moving model to device:  mps


HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint()
  (blocks): ModuleList(
    (0-23): 24 x TransformerBlock(
      (ln1): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (ln2): RMSNormPre(
        (hook_scale): HookPoint()
        (hook_normalized): HookPoint()
      )
      (attn): GroupedQueryAttention(
        (hook_k): HookPoint()
        (hook_q): HookPoint()
        (hook_v): HookPoint()
        (hook_z): HookPoint()
        (hook_attn_scores): HookPoint()
        (hook_pattern): HookPoint()
        (hook_result): HookPoint()
        (hook_rot_k): HookPoint()
        (hook_rot_q): HookPoint()
      )
      (mlp): GatedMLP(
        (hook_pre): HookPoint()
        (hook_pre_linear): HookPoint()
        (hook_post): HookPoint()
      )
      (hook_attn_in): HookPoint()
      (hook_q_input): HookPoint()
      (hook_k_input): HookPoint()
      (hook_v_input): HookPoint()
      (hook_mlp_in): HookPoint()
      (hook_att

## Find the data with the correct answer

In [ ]:
from src import filter_correct_data
import pandas as pd

dataset_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_corrected.csv"
filtered_data_path = "hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered.csv"
test_data = pd.read_csv(dataset_path)

filtered_data = filter_correct_data(model, test_data, "original_sentence", "original_triplet", save_path=filtered_data_path)

## Create EAP Dataset

#### Building the Dataset

In [10]:

from src.utils import build_eap_dataset
import pandas as pd


filtered_data = pd.read_csv("hotel_dataset/hotel_aste_model-Qwen2.5-0.5B_lr-0.0001_bs-16_epochs-20_counterfactual_filtered.csv")

eap_df = build_eap_dataset(
    model=model,
    df=filtered_data,
    sentence_col="original_sentence",
    triplet_col="original_triplet",
    corrupted_col="counterfact1_modified",
    corrupted_triplet_col="counterfact_triplet1_modified",
    suffix="[A]",
    idx=0  # 0 for aspect, 1 for opinion, 2 for sentiment
)

eap_df.to_csv("eap_dataset/eap_dataset_aspect.csv", index=False)

# EAP-IG

In [1]:
from functools import partial

import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PreTrainedTokenizer
from transformer_lens import HookedTransformer

from eap.graph import Graph
from eap.evaluate import evaluate_graph, evaluate_baseline
from eap.attribute import attribute 
from datasets import Dataset
import ast

from src import load_finetuned_model

In [ ]:
import ast
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch

def safe_parse(raw):
    try:
        return ast.literal_eval(str(raw))[0]  # unbox the list-of-list
    except Exception as e:
        raise ValueError(f"Failed to parse: {raw}\n{e}")

class EAPDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        clean = row["clean"]
        corrupted = row["corrupted"]
        correct_idx = safe_parse(row["correct_idx"])
        incorrect_idx = safe_parse(row["incorrect_idx"])
        return clean, corrupted, [correct_idx, incorrect_idx]

def collate_EAP(batch):
    clean, corrupted, labels = zip(*batch)

    correct_idx_batch = [torch.tensor(l[0], dtype=torch.long) for l in labels]
    incorrect_idx_batch = [torch.tensor(l[1], dtype=torch.long) for l in labels]

    return list(clean), list(corrupted), (correct_idx_batch, incorrect_idx_batch)

    
def get_logit_positions(logits: torch.Tensor, input_length: torch.Tensor):
    batch_size = logits.size(0)
    idx = torch.arange(batch_size, device=logits.device)

    logits = logits[idx, input_length - 1]
    return logits

def logit_diff(logits: torch.Tensor, clean_logits: torch.Tensor, input_length: torch.Tensor, labels: torch.Tensor, mean=True, loss=False):
    logits = get_logit_positions(logits, input_length)
    good_bad = torch.gather(logits, -1, labels.to(logits.device))
    results = good_bad[:, 0] - good_bad[:, 1]
    if loss:
        results = -results
    if mean: 
        results = results.mean()
    return results

def logit_diff_multitoken(logits: torch.Tensor, clean_logits: torch.Tensor, input_lengths: torch.Tensor, labels: tuple, mean=True, loss=False):
    correct_indices_batch, incorrect_indices_batch = labels  # Each is a list of tensors (batch_size length)

    results = []
    for i in range(len(correct_indices_batch)):
        correct_ids = correct_indices_batch[i]  # Tensor of token ids
        incorrect_ids = incorrect_indices_batch[i]

        # Grab logits at the last sequence position for that sample
        # (logits shape: [batch_size, seq_len, vocab_size])
        final_logits = logits[i, -1]  # [vocab_size]

        correct_score = final_logits[correct_ids].mean()  # or .sum()
        incorrect_score = final_logits[incorrect_ids].mean()

        diff = correct_score - incorrect_score
        if loss:
            diff = -diff
        results.append(diff)

    output = torch.stack(results)
    return output.mean() if mean else output


In [3]:
base_model_name = "Qwen/Qwen2.5-0.5B"
fine_tuned_model_path = "models/fine_tuned_model/"
model = load_finetuned_model(base_model_name, fine_tuned_model_path)
model.to("mps")
model.cfg.use_split_qkv_input = True
model.cfg.use_attn_result = True
model.cfg.use_hook_mlp_in = True
model.cfg.ungroup_grouped_query_attention = True

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


Moving model to device:  mps


In [28]:
ds = EAPDataset("eap_dataset/eap_dataset_aspect.csv")
dataloader = DataLoader(ds, batch_size=10, collate_fn=collate_EAP)

In [22]:
# Instantiate a graph with a model
g = Graph.from_model(model)

In [29]:
attribute(
    model,
    g,
    dataloader,
    partial(logit_diff_multitoken, loss=True, mean=True),
    method='EAP-IG-inputs',
    ig_steps=5,
    is_absa=True,
    absa_element="A"  # or "O", "S"
)

  0%|          | 0/6 [00:03<?, ?it/s]

Number of positions must match, but do not: 24 (clean) != 22 (corrupted)
['airnya kurang kencang . [A]', 'tidak ada fasilitas airy , . [A]', 'air kamar mandinnya asin . [A]', 'menjadi pengalaman yang sangat berkesan . [A]', 'parkir mobil agak susah . [A]', 'memang tidak ada sarapan , . [A]', 'saya sudah 4 kali menginap di sini , overall pelayanannya baik . [A]', 'tidak ada sabun di dalam kamar . [A]', 'wifi nya kurang memuaskan : ) . [A]', 'kamar sempit . [A]']
['bulannya kurang kencang . [A]', 'tidak ada lingkungan , . [A]', 'ideologinya asin . [A]', 'menjadi gajah yang sangat berkesan . [A]', 'pertanian agak susah . [A]', 'memang tidak ada gelap , . [A]', 'saya sudah 4 kali menginap di sini , konstituen baik . [A]', 'tidak ada astronomi di dalam kamar . [A]', 'astronot kurang memuaskan : ) . [A]', 'anginnya sempit . [A]']


ValueError: Number of positions must match